In [ ]:
from huggingface_hub import login

login()

In [ ]:
!pip -q install bert-score evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [ ]:
import pandas as pd

eval_df = pd.read_csv(
    "/content/drive/MyDrive/llm_benchmark/persian_chatbot_eval_1000.csv"
)

pilot_df = eval_df.iloc[:1000].copy()

print(pilot_df.shape)
pilot_df.head()

(1000, 3)


,sample_id,inputs,outputs
0,0,در این مسئله متن را کامل بخوان و بهترین جواب ر...,شهر گواتمالاسیتی
1,1,در این مسئله متن را کامل بخوان و بهترین جواب ر...,زوال عقل سالخوردگی یا بیماری آلزایمر
2,2,در این مسئله متن را کامل بخوان و بهترین جواب ر...,جرماغون نویان
3,3,در این مسئله متن را کامل بخوان و بهترین جواب ر...,برای نخستین بار در ساخت آن، هم از قطعات مکانیک...
4,4,در این مسئله متن را کامل بخوان و بهترین جواب ر...,ازدواج دخترش تاج الملوک با مظفرالدین شاه قاجار...


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "PartAI/Dorna2-Llama3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

print("Loaded successfully")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.3k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded successfully


In [ ]:
print("Loaded:", MODEL_NAME)

Loaded: PartAI/Dorna2-Llama3.1-8B-Instruct


In [ ]:
def generate_answer(question):

    messages = [
        {
            "role": "system",
            "content": """
شما یک سامانه پرسش و پاسخ فارسی هستید.

قوانین:
- فقط پاسخ نهایی را بنویس.
- توضیح نده.
- استدلال نکن.
- مقدمه ننویس.
- فقط خود پاسخ را برگردان.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [ ]:
for q in [
    "پایتخت ایران چیست؟",
    "بزرگترین سیاره منظومه شمسی چیست؟",
    "معروف ترین نوع زوال عقل چیست؟"
]:
    print("Q:", q)
    print("A:", generate_answer(q))
    print("-"*50)

Q: پایتخت ایران چیست؟
A: تهران
--------------------------------------------------
Q: بزرگترین سیاره منظومه شمسی چیست؟
A: ژوپیتر
--------------------------------------------------
Q: معروف ترین نوع زوال عقل چیست؟
A: زوال عقل نوعی بیماری عصبی است که بر توانایی‌های شناختی و رفتاری فرد تاثیر می‌گذ
--------------------------------------------------


In [ ]:
print(model.quantization_method if hasattr(model, "quantization_method") else "No quantization_method")

QuantizationMethod.BITS_AND_BYTES


In [ ]:
for name, module in model.named_modules():
    if "Linear4bit" in str(type(module)):
        print("Found 4-bit layer:", name)
        break

Found 4-bit layer: model.layers.0.self_attn.q_proj


In [ ]:
print(model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": fa

In [ ]:
import time
import torch
import pandas as pd

predictions = []
references = []

latencies = []
token_counts = []

torch.cuda.reset_peak_memory_stats()

for idx, row in pilot_df.iterrows():

    prompt = row["inputs"]
    reference = row["outputs"]

    start = time.time()

    prediction = generate_answer(prompt)

    latency = time.time() - start

    pred_tokens = len(
        tokenizer.encode(
            prediction,
            add_special_tokens=False
        )
    )

    predictions.append(prediction)
    references.append(reference)

    latencies.append(latency)
    token_counts.append(pred_tokens)

    if (idx + 1) % 50 == 0:

        checkpoint_df = pd.DataFrame({
            "reference": references,
            "prediction": predictions,
            "latency": latencies,
            "tokens": token_counts
        })

        checkpoint_df.to_csv(
            "/content/drive/MyDrive/llm_benchmark/dorna2_partial.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Checkpoint saved: {idx+1}")

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx+1}/1000")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Completed 10/1000
Completed 20/1000
Completed 30/1000
Completed 40/1000
Checkpoint saved: 50
Completed 50/1000
Completed 60/1000
Completed 70/1000
Completed 80/1000
Completed 90/1000
Checkpoint saved: 100
Completed 100/1000
Completed 110/1000
Completed 120/1000
Completed 130/1000
Completed 140/1000
Checkpoint saved: 150
Completed 150/1000
Completed 160/1000
Completed 170/1000
Completed 180/1000
Completed 190/1000
Checkpoint saved: 200
Completed 200/1000
Completed 210/1000
Completed 220/1000
Completed 230/1000
Completed 240/1000
Checkpoint saved: 250
Completed 250/1000
Completed 260/1000
Completed 270/1000
Completed 280/1000
Completed 290/1000
Checkpoint saved: 300
Completed 300/1000
Completed 310/1000
Completed 320/1000
Completed 330/1000
Completed 340/1000
Checkpoint saved: 350
Completed 350/1000
Completed 360/1000
Completed 370/1000
Completed 380/1000
Completed 390/1000
Checkpoint saved: 400
Completed 400/1000
Completed 410/1000
Completed 420/1000
Completed 430/1000
Completed 440/100

In [ ]:
results_df = pd.DataFrame({
    "reference": references,
    "prediction": predictions,
    "latency": latencies,
    "tokens": token_counts
})

results_df.to_csv(
    "/content/drive/MyDrive/llm_benchmark/dorna2_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved.")

Results saved.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

pred_emb = sim_model.encode(
    predictions,
    batch_size=32,
    show_progress_bar=True
)

ref_emb = sim_model.encode(
    references,
    batch_size=32,
    show_progress_bar=True
)

similarities = [
    cosine_similarity(
        pred.reshape(1, -1),
        ref.reshape(1, -1)
    )[0][0]
    for pred, ref in zip(pred_emb, ref_emb)
]

semantic_similarity = float(np.mean(similarities))

print("Semantic Similarity:", semantic_similarity)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Semantic Similarity: 0.7292196154594421


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

rouge_l = rouge_result["rougeL"]

print("ROUGE-L:", rouge_l)

ROUGE-L: 0.010833333333333332


In [ ]:
avg_latency = sum(latencies) / len(latencies)

total_tokens = sum(token_counts)

total_time = sum(latencies)

tokens_per_sec = total_tokens / total_time

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("Average Latency:", avg_latency)
print("Tokens/sec:", tokens_per_sec)
print("Peak VRAM:", peak_vram)

Average Latency: 2.520556212425232
Tokens/sec: 8.552477383259093
Peak VRAM: 6.474133491516113


In [ ]:
summary_df = pd.DataFrame({
    "model": ["Dorna2-Llama3.1-8B-Instruct"],
    "num_samples": [len(predictions)],
    "semantic_similarity": [semantic_similarity],
    "rouge_l": [rouge_l],
    "avg_latency_sec": [avg_latency],
    "tokens_per_sec": [tokens_per_sec],
    "peak_vram_gb": [peak_vram]
})

summary_df.to_csv(
    "/content/drive/MyDrive/llm_benchmark/dorna2_summary.csv",
    index=False
)

summary_df

,model,num_samples,semantic_similarity,rouge_l,avg_latency_sec,tokens_per_sec,peak_vram_gb
0,Dorna2-Llama3.1-8B-Instruct,1000,0.72922,0.010833,2.520556,8.552477,6.474133
